In [ ]:
import torch
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import DataLoader
from os.path import join
from PIL import Image

from data_load_BT import CovidDataset

# Our own DL models.
from models1 import VAE




def dice_coefficient(pred_mask, true_mask, smooth=1e-6):
    """
    이진 분류를 위한 Dice 계수 계산
    """
    # reshape 사용 (view 대신)
    pred_flat = pred_mask.reshape(-1)
    target_flat = true_mask.reshape(-1)
    
    intersection = torch.sum(pred_flat * target_flat)
    return (2.0 * intersection + smooth) / (torch.sum(pred_flat) + torch.sum(target_flat) + smooth)


def calculate_binary_dice_score(pred, target, threshold=0.5):
    """
    pred: [B, 1, H, W] - 모델 출력 (logit)
    target: [B, 1, H, W] - 단일 채널 마스크 (0/1 or 0/255)
    """
    # 1) 로짓 -> 시그모이드 후 임계값
    pred_prob = torch.sigmoid(pred)
    pred_binary = (pred_prob > threshold).float()

    # 2) GT를 확실히 이진화 (0/1이든 0/255든 안전)
    #    값 범위가 [0,1]이면 0.5, [0,255]면 127.5 근사. 아래처럼 0.5 기준으로 처리하고,
    #    필요하면 데이터셋에 맞춰 127.5로 바꾸세요.s
    if target.dtype != torch.float32:
        target = target.float()

    # 타깃이 0/255일 가능성 대비: (최댓값이 1보다 크면 127.5 기준으로, 아니면 0.5 기준)
    tmax = torch.amax(target)
    thr = 127.5 if tmax > 1.5 else 0.5
    target_bin = (target > thr).float()

    # contiguous (안전)
    target_bin = target_bin.contiguous()
    pred_binary = pred_binary.contiguous()

    dice = dice_coefficient(pred_binary, target_bin).item()
    return {'Foreground': dice}



def save_segmentation_results(images, targets, outputs, batch_idx, batch_size, output_dir, dataset, threshold=0.5):
    """
    이진 세그멘테이션 결과를 이미지로 저장하는 함수
    """
    # 디렉토리 없으면 생성
    os.makedirs(os.path.join(output_dir, 'input'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'target'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'prediction'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'overlay'), exist_ok=True)
    
    # 예측 결과 준비
    pred_sigmoid = torch.sigmoid(outputs)
    pred_binary = (pred_sigmoid > threshold).float().cpu().numpy()  # (B, 1, H, W)
    
    # 배치 내 이미지 처리
    for i in range(images.size(0)):
        idx = batch_idx * batch_size + i
        
        if idx >= len(dataset):
            break
        
        # 원본 이미지 (입력이 단일 채널인 경우)
        input_img = images[i, 0].cpu().numpy()
        input_img = (input_img * 255).astype(np.uint8)
        
        # 타겟 마스크 (전경 클래스만)
        target_mask = targets[i, 0].cpu().numpy()  # [H, W] - 단일 채널 마스크
        
        # 예측 마스크
        pred_mask = pred_binary[i, 0]  # [H, W]
        
        # 이진 클래스 색상 맵 정의
        colors = [
            [0, 0, 0],       # 배경: 검정
            [255, 0, 0],     # 전경: 빨강
        ]
        
        # 컬러 마스크 생성
        target_rgb = np.zeros((target_mask.shape[0], target_mask.shape[1], 3), dtype=np.uint8)
        pred_rgb = np.zeros((pred_mask.shape[0], pred_mask.shape[1], 3), dtype=np.uint8)
        
        target_rgb[target_mask > 0] = colors[1]  # 전경은 빨간색
        pred_rgb[pred_mask > 0] = colors[1]  # 전경은 빨간색
        
        # 입력 이미지를 3채널로 변환
        input_rgb = np.stack([input_img] * 3, axis=2)
        
        # 오버레이 이미지 생성 (입력 + 예측 마스크)
        overlay = input_rgb.copy()
        # 전경 부위만 반투명하게 오버레이
        mask = pred_mask > 0
        overlay[mask] = (overlay[mask] * 0.3 + pred_rgb[mask] * 0.7).astype(np.uint8)
        
        # 이미지 저장
        filename = f"image_{idx:04d}"
        Image.fromarray(input_img).save(os.path.join(output_dir, 'input', f"{filename}.png"))
        Image.fromarray(target_rgb).save(os.path.join(output_dir, 'target', f"{filename}.png"))
        Image.fromarray(pred_rgb).save(os.path.join(output_dir, 'prediction', f"{filename}.png"))
        Image.fromarray(overlay).save(os.path.join(output_dir, 'overlay', f"{filename}.png"))

def evaluate_test_set(model, test_loader, device, save_segmentation=False, output_dir=None, dataset=None, threshold=0.5):
    """
    이진 분류 테스트 데이터셋으로 모델을 평가하고 Dice Score 반환
    """
    model.eval()
    dice_scores = []
    
    if save_segmentation and output_dir is not None:
        os.makedirs(output_dir, exist_ok=True)
    
    print(f"테스트 시작... 총 {len(test_loader)} 개의 배치")
    
    with torch.no_grad():
        for batch_idx, batch_data in enumerate(tqdm(test_loader)):
            images, targets,_,_ = batch_data

            images = images.unsqueeze(1)
            targets = targets.unsqueeze(1)

            
            
            # 데이터를 디바이스로 이동
            images = images.to(device)
            targets = targets.to(device)

            
            
            try:
                # 모델 추론
                outputs = model(images)
                if isinstance(outputs, tuple):
                    outputs = outputs[0]  # 모델이 여러 출력을 반환하는 경우
                
                # 디버깅 정보 출력 (첫 배치만)
                if batch_idx == 0:
                    print(f"Model output shape: {outputs.shape}")
                    print(f"Target shape: {targets.shape}")
                    print(f"Target foreground shape: {targets[:, 1:2].shape}")
                
                # Dice 점수 계산
                dice_score = calculate_binary_dice_score(outputs, targets, threshold)
                dice_scores.append(dice_score['Foreground'])
                
                # 세그멘테이션 결과 저장
                if save_segmentation and output_dir is not None:
                    save_segmentation_results(images, targets, outputs, batch_idx, test_loader.batch_size, output_dir, dataset, threshold)
                
            except Exception as e:
                print(f"배치 {batch_idx} 처리 중 오류 발생: {e}")
                import traceback
                traceback.print_exc()
                if batch_idx == 0:
                    raise
    
    mean_dice = np.mean(dice_scores)
    
    # 전체 Dice 점수를 파일로 저장
    if output_dir is not None:
        with open(os.path.join(output_dir, 'dice_scores.txt'), 'w') as f:
            f.write(f"전경 Dice Score: {mean_dice:.4f}\n")
    
    return mean_dice


def main():
    test_dir = 'datasets/covid_8_1_1/test'
    model_path = 'results/checkpoints/VAE_model_best_withoutPT_100.pt'
    gpu_id = 0
    batch_size = 32
    save_segmentation = True
    output_dir = 'results/result_image_WP_100'
    threshold = 0.5  # 이진 분류를 위한 임계값
    
    device = "cuda:2" if torch.cuda.is_available() else "cpu"
    try:
        # 모델 로드
        seg_model = VAE(base=32).to(device)
        
        # 체크포인트 로드 (불일치하는 레이어는 무시)
        checkpoint = torch.load(model_path, map_location=device)
        
        # final_conv 레이어의 최종 컨볼루션 가중치와 편향 제거

        
        # 수정된 체크포인트 로드 (strict=False로 설정)
        seg_model.load_state_dict(checkpoint, strict=False)
        
        print(f"모델을 로드했습니다: {model_path}")
    except Exception as e:
        print(f"모델 로드 중 오류 발생: {e}")
        import traceback
        traceback.print_exc()
        return
    
    try:
        # 테스트 데이터셋 로드
        test_dataset = CovidDataset(test_dir)
        
        test_loader = DataLoader(
            test_dataset, 
            batch_size=batch_size, 
            shuffle=False,
            num_workers=4,
            pin_memory = device.startswith("cuda")
        )
        
        print(f"테스트 데이터셋 로드 완료: {len(test_dataset)} 이미지")
    except Exception as e:
        print(f"데이터셋 로드 중 오류 발생: {e}")
        import traceback
        traceback.print_exc()
        return
    
    try:
        # 모델 평가
        mean_dice = evaluate_test_set(
            seg_model, test_loader, device, 
            save_segmentation, output_dir, test_dataset, 
            threshold
        )
        
        print("\n" + "="*50)
        print(f"전경 Dice Score: {mean_dice:.4f}")
        print("="*50)
        
        if save_segmentation:
            print(f"\n세그멘테이션 결과 저장 완료: {output_dir}")
    except Exception as e:
        print(f"평가 중 오류 발생: {e}")
        import tracebackS
        traceback.print_exc()


if __name__ == "__main__":
    main()

In [ ]:
import torch
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import DataLoader
from os.path import join
from PIL import Image

from data_load_BT import CovidDataset

# Our own DL models.
from models1 import VAE


def dice_coefficient(pred_mask, true_mask, smooth=1e-6):
    """
    이진 분류를 위한 Dice 계수 계산
    """
    # reshape 사용 (view 대신)
    pred_flat = pred_mask.reshape(-1)
    target_flat = true_mask.reshape(-1)
    
    intersection = torch.sum(pred_flat * target_flat)
    return (2.0 * intersection + smooth) / (torch.sum(pred_flat) + torch.sum(target_flat) + smooth)


def calculate_metrics(pred, target, threshold=0.5):
    """
    pred: [B, 1, H, W]
    target: [B, 1, H, W]
    threshold: float
    """

    # sigmoid + threshold
    pred_prob = torch.sigmoid(pred)
    pred_bin = (pred_prob > threshold).float()

    if target.dtype != torch.float32:
        target = target.float()

    # target 이 0/255일 가능성 처리
    thr = 127.5 if torch.amax(target) > 1.5 else 0.5
    target_bin = (target > thr).float()

    pred_bin = pred_bin.contiguous()
    target_bin = target_bin.contiguous()

    # Flatten
    p = pred_bin.view(-1)
    t = target_bin.view(-1)

    # Dice
    intersection = (p * t).sum()
    dice = (2 * intersection) / (p.sum() + t.sum() + 1e-6)

    # IoU
    union = p.sum() + t.sum() - intersection
    iou = intersection / (union + 1e-6)

    # Precision
    precision = intersection / (p.sum() + 1e-6)

    #Recall
    recall = intersection / (t.sum() + 1e-6)

    return {
        "dice": dice.item(),
        "iou": iou.item(),
        "precision": precision.item(),
        "recall": recall.item()
    }


# === 세그멘테이션 이미지 저장 함수는 제거함 ===
# def save_segmentation_results(...):
#     ...


def evaluate_test_set(model, test_loader, device, save_segmentation=False, output_dir=None, dataset=None, threshold=0.5):
    """
    이진 분류 테스트 데이터셋으로 모델을 평가하고 Dice / mIoU / Precision 반환
    """
    model.eval()
        
    dice_scores = []
    iou_scores = []
    precision_scores = []
    recall_scores = []

    # (이미지 저장은 더 이상 하지 않으므로 여기서는 디렉토리 생성 안 함)
    
    print(f"테스트 시작... 총 {len(test_loader)} 개의 배치")
    
    with torch.no_grad():
        for batch_idx, batch_data in enumerate(tqdm(test_loader)):
            images, targets, _, _ = batch_data

            images = images.unsqueeze(1)
            targets = targets.unsqueeze(1)
            
            # 데이터를 디바이스로 이동
            images = images.to(device)
            targets = targets.to(device)
            
            try:
                # 모델 추론
                outputs = model(images)
                if isinstance(outputs, tuple):
                    outputs = outputs[0]  # 모델이 여러 출력을 반환하는 경우
                
                # 디버깅 정보 출력 (첫 배치만)
                if batch_idx == 0:
                    print(f"Model output shape: {outputs.shape}")
                    print(f"Target shape: {targets.shape}")
                    print(f"Target foreground shape: {targets[:, 1:2].shape}")
                
                # 메트릭 계산
                metrics = calculate_metrics(outputs, targets, threshold)

                dice_scores.append(metrics["dice"])
                iou_scores.append(metrics["iou"])
                precision_scores.append(metrics["precision"])
                recall_scores.append(metrics["recall"])

                # 세그멘테이션 결과 저장 부분은 제거됨

            except Exception as e:
                print(f"배치 {batch_idx} 처리 중 오류 발생: {e}")
                import traceback
                traceback.print_exc()
                if batch_idx == 0:
                    raise
    
    mean_dice = np.mean(dice_scores)
    mean_iou = np.mean(iou_scores)
    mean_precision = np.mean(precision_scores)
    mean_recall = np.mean(recall_scores)
    
    print("\n" + "="*50)
    print(f"Foreground Dice Score: {mean_dice:.4f}")
    print(f"Foreground mIoU:       {mean_iou:.4f}")
    print(f"Foreground Precision:  {mean_precision:.4f}")
    print(f"Foreground Recall:     {mean_recall:.4f}")
    print("="*50)

    # === 점수들을 파일로 저장 ===
    if output_dir is not None:
        os.makedirs(output_dir, exist_ok=True)   # 폴더 없으면 생성
        metrics_path = os.path.join(output_dir, "metrics.txt")
        with open(metrics_path, "w") as f:
            f.write(f"Foreground Dice Score: {mean_dice:.4f}\n")
            f.write(f"Foreground mIoU:       {mean_iou:.4f}\n")
            f.write(f"Foreground Precision:  {mean_precision:.4f}\n")
            f.write(f"Foreground Recall:     {mean_recall:.4f}\n")
        print(f"\n메트릭 저장 완료: {metrics_path}")
    
    return mean_dice, mean_iou, mean_precision


def main():
    test_dir = 'datasets/covid_8_1_1/test'
    model_path = 'results/checkpoints/VAE_model_best_withoutPT_30.pt'
    gpu_id = 0
    batch_size = 32
    save_segmentation = True  # 이제는 의미 없음 (이미지 저장 안 함)
    output_dir = 'results/result_image_WP_30'
    threshold = 0.5  # 이진 분류를 위한 임계값
    
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    try:
        # 모델 로드
        seg_model = VAE(base=32).to(device)
        
        # 체크포인트 로드 (불일치하는 레이어는 무시)
        checkpoint = torch.load(model_path, map_location=device)
        
        # 수정된 체크포인트 로드 (strict=False로 설정)
        seg_model.load_state_dict(checkpoint, strict=False)
        
        print(f"모델을 로드했습니다: {model_path}")
    except Exception as e:
        print(f"모델 로드 중 오류 발생: {e}")
        import traceback
        traceback.print_exc()
        return
    
    try:
        # 테스트 데이터셋 로드
        test_dataset = CovidDataset(test_dir)
        
        test_loader = DataLoader(
            test_dataset, 
            batch_size=batch_size, 
            shuffle=False,
            num_workers=4,
            pin_memory = device.startswith("cuda")
        )
        
        print(f"테스트 데이터셋 로드 완료: {len(test_dataset)} 이미지")
    except Exception as e:
        print(f"데이터셋 로드 중 오류 발생: {e}")
        import traceback
        traceback.print_exc()
        return
    
    try:
        # 모델 평가
        mean_dice, mean_iou, mean_precision = evaluate_test_set(
            seg_model, test_loader, device, 
            save_segmentation, output_dir, test_dataset, 
            threshold
        )
        
        
        
        # 세그멘테이션 이미지는 저장하지 않지만,
        # output_dir 아래 metrics.txt 로 점수는 저장됨.
    except Exception as e:
        print(f"평가 중 오류 발생: {e}")
        import tracebackS
        traceback.print_exc()


if __name__ == "__main__":
    main()


In [ ]:
import torch
import numpy as np
import os
import time               # ★ 인퍼런스 타임 측정용
from tqdm import tqdm
from torch.utils.data import DataLoader

from data_load_BT import CovidDataset
from models1 import VAE


def dice_coefficient(pred_mask, true_mask, smooth=1e-6):
    """
    이진 분류를 위한 Dice 계수 계산
    """
    pred_flat = pred_mask.reshape(-1)
    target_flat = true_mask.reshape(-1)
    
    intersection = torch.sum(pred_flat * target_flat)
    return (2.0 * intersection + smooth) / (torch.sum(pred_flat) + torch.sum(target_flat) + smooth)


def calculate_binary_dice_score(pred, target, threshold=0.5):
    """
    pred: [B, 1, H, W] - 모델 출력 (logit)
    target: [B, 1, H, W] - 단일 채널 마스크 (0/1 or 0/255)
    """
    # 1) 로짓 -> 시그모이드 후 임계값
    pred_prob = torch.sigmoid(pred)
    pred_binary = (pred_prob > threshold).float()

    # 2) GT를 확실히 이진화
    if target.dtype != torch.float32:
        target = target.float()

    tmax = torch.amax(target)
    thr = 127.5 if tmax > 1.5 else 0.5
    target_bin = (target > thr).float()

    target_bin = target_bin.contiguous()
    pred_binary = pred_binary.contiguous()

    dice = dice_coefficient(pred_binary, target_bin).item()
    return {'Foreground': dice}


def evaluate_test_set(model, test_loader, device, threshold=0.5):
    """
    이진 분류 테스트 데이터셋으로 모델을 평가하고
    - Dice Score
    - 앞 10장은 warmup으로 빼고, 나머지에 대한 1장당 평균 inference time (초)
    를 반환.
    """
    model.eval()
    dice_scores = []

    print(f"테스트 시작... 총 {len(test_loader)} 개의 배치")
    
    warmup_images = 10          # ★ 앞 10장은 로드/워밍업 구간으로 취급
    seen_images = 0             # 지금까지 처리한 전체 이미지 수
    total_forward_time = 0.0    # 전체 forward 시간 합
    warmup_forward_time = 0.0   # 그 중 warmup(앞 10장)이 차지하는 시간 합

    with torch.no_grad():
        for batch_idx, batch_data in enumerate(tqdm(test_loader)):
            images, targets, _, _ = batch_data

            images = images.unsqueeze(1)   # (B, 1, H, W)
            targets = targets.unsqueeze(1) # (B, 1, H, W)

            images = images.to(device)
            targets = targets.to(device)

            # if str(device).startswith("cuda"):
            #     images = images.half()

            B = images.size(0)
            batch_start_idx = seen_images
            batch_end_idx = seen_images + B  # [batch_start_idx, batch_end_idx)

            try:
                # ====== forward 시간 측정 ======
                if str(device).startswith("cuda"):
                    torch.cuda.synchronize()
                t0 = time.time()

                outputs = model(images)

                if str(device).startswith("cuda"):
                    torch.cuda.synchronize()
                forward_time = time.time() - t0

                total_forward_time += forward_time

                if isinstance(outputs, tuple):
                    outputs = outputs[0]

                # 디버깅 정보 (첫 배치만)
                if batch_idx == 0:
                    print(f"Model output shape: {outputs.shape}")
                    print(f"Target shape: {targets.shape}")
                    # targets[:, 1:2]는 채널 2개일 때만 의미 있어서 제거함

                # ===== warmup / 측정 구간 분리 =====
                warmup_count = 0
                if batch_start_idx < warmup_images:
                    warmup_count = min(warmup_images - batch_start_idx, B)
                valid_count = B - warmup_count

                if warmup_count > 0:
                    warmup_forward_time += forward_time * (warmup_count / B)

                # Dice 계산 (전체 배치 기준으로)
                dice_score = calculate_binary_dice_score(outputs, targets, threshold)
                dice_scores.append(dice_score['Foreground'])

                seen_images += B

            except Exception as e:
                print(f"배치 {batch_idx} 처리 중 오류 발생: {e}")
                import traceback
                traceback.print_exc()
                if batch_idx == 0:
                    raise
    
    mean_dice = np.mean(dice_scores)

    # ===== 인퍼런스 타임 정리 =====
    total_images = seen_images
    measured_images = max(total_images - warmup_images, 0)
    measured_forward_time = total_forward_time - warmup_forward_time

    if measured_images > 0:
        avg_infer_time_per_image = measured_forward_time / measured_images  # 초 단위
    else:
        avg_infer_time_per_image = 0.0

    # print("seen images: ",seen_images)

    return mean_dice, avg_infer_time_per_image


# ★ 모델 크기/파라미터 정보 출력 함수 추가
def print_model_info(model, model_path=None):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print("\n===== 모델 정보 =====")
    print(f"총 파라미터 수: {total_params:,}")
    print(f"학습 가능한 파라미터 수: {trainable_params:,}")

    if model_path is not None and os.path.exists(model_path):
        file_size_mb = os.path.getsize(model_path) / (1024 ** 2)
        print(f"모델 파일 용량: {file_size_mb:.2f} MB")
    print("=====================\n")


def main():
    test_dir = 'datasets/covid_8_1_1/test'
    model_path = 'results/checkpoints/VAE_model_best_30.pt'
    gpu_id = 0
    batch_size = 32
    threshold = 0.5  # 이진 분류를 위한 임계값
    
    # device = "cuda:2" if torch.cuda.is_available() else "cpu"
    device = "cpu"
    print(f"사용 디바이스: {device}")

    try:
        # 모델 로드
        seg_model = VAE(base=32).to(device)

        checkpoint = torch.load(model_path, map_location=device)
        seg_model.load_state_dict(checkpoint, strict=False)

        # if str(device).startswith("cuda"):
        #     seg_model.half()

        # ★ 여기서 모델 크기/파라미터 정보 출력
        print_model_info(seg_model, model_path)

        print(f"모델을 로드했습니다: {model_path}")
    except Exception as e:
        print(f"모델 로드 중 오류 발생: {e}")
        import traceback
        traceback.print_exc()
        return
    
    try:
        # 테스트 데이터셋 로드
        test_dataset = CovidDataset(test_dir)
        
        test_loader = DataLoader(
            test_dataset, 
            batch_size=batch_size, 
            shuffle=False,
            num_workers=4,
            pin_memory=device.startswith("cuda")
        )
        
        print(f"테스트 데이터셋 로드 완료: {len(test_dataset)} 이미지")
    except Exception as e:
        print(f"데이터셋 로드 중 오류 발생: {e}")
        import traceback
        traceback.print_exc()
        return
    
    try:
        # 모델 평가 (Dice + 인퍼런스 타임)
        mean_dice, avg_infer_time = evaluate_test_set(
            seg_model, test_loader, device, threshold
        )
        
        print("\n" + "="*50)
        print(f"전경 Dice Score: {mean_dice:.4f}")
        print(f"평균 인퍼런스 타임 (1장당, warmup 10장 제외): {avg_infer_time*1000:.3f} ms")
        print("="*50)
        

    except Exception as e:
        print(f"평가 중 오류 발생: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    main()


In [ ]:
# import torch
# from models1 import VAE

# model_path = "results/checkpoints/VAE_model_best_30.pt"
# model_fp16_path = "results/checkpoints/VAE_model_best_30_fp16.pt"

# device = "cpu"

# # 1) 원래 모델 로드
# model = VAE(base=32).to(device)
# checkpoint = torch.load(model_path, map_location=device)
# model.load_state_dict(checkpoint, strict=False)

# # 2) half로 변환
# model.half()

# # 3) state_dict만 다시 저장 (이제 float16 타입으로 저장됨)
# torch.save(model.state_dict(), model_fp16_path)

# print("원본 파일 크기:", os.path.getsize(model_path) / (1024**2), "MB")
# print("fp16 파일 크기:", os.path.getsize(model_fp16_path) / (1024**2), "MB")






In [ ]:
import os
from os.path import join
import numpy as np
from PIL import Image
from tqdm import tqdm

# === 경로 설정 ===
pred_dir = "results/mist_overlay/pred"
tgt_dir  = "results/mist_overlay/gt"
out_dir  = "results/mist_overlay/overlay"

os.makedirs(out_dir, exist_ok=True)

# prediction 폴더 기준으로 파일 이름 수집 (png만)
pred_files = sorted([f for f in os.listdir(pred_dir) if f.lower().endswith(".png")])

for fname in tqdm(pred_files, desc="Overlay generating"):
    pred_path = join(pred_dir, fname)
    tgt_path  = join(tgt_dir, fname)

    # target에 같은 이름이 없으면 스킵
    if not os.path.exists(tgt_path):
        print(f"[WARN] target not found for {fname}, skip")
        continue

    # === 이미지 로드 (그레이스케일로) ===
    pred_img = Image.open(pred_path).convert("L")
    tgt_img  = Image.open(tgt_path).convert("L")

    pred_np = np.array(pred_img)
    tgt_np  = np.array(tgt_img)

    # 크기가 다르면 스킵 (원하면 resize로 맞추게 바꿀 수 있음)
    if pred_np.shape != tgt_np.shape:
        print(f"[WARN] shape mismatch for {fname}: pred{pred_np.shape}, tgt{tgt_np.shape}")
        continue

    # === 이진 마스크로 변환 ===
    # 0/255 마스크라고 가정하고, 0이 아닌 픽셀은 전부 1로 처리
    pred_bin = (pred_np > 0)
    tgt_bin  = (tgt_np > 0)

    # === TP / FP / FN 계산 ===
    tp = pred_bin & tgt_bin          # 맞게 예측한 부분 (1 & 1)
    fp = pred_bin & (~tgt_bin)       # prediction만 1인 부분 (잘못 예측, False Positive)
    fn = (~pred_bin) & tgt_bin       # target만 1인 부분 (놓친 곳, False Negative)

    h, w = pred_np.shape
    overlay = np.zeros((h, w, 3), dtype=np.uint8)  # RGB

    # 색 지정
    overlay[tp] = [0, 255, 0]        # 초록 (TP)
    overlay[fp] = [255, 0, 0]        # 빨강 (FP)
    overlay[fn] = [255, 255, 255]    # 흰색 (FN)
    # 나머지(TN)는 검정 [0,0,0] 그대로

    # === 저장 ===
    out_path = join(out_dir, fname)
    Image.fromarray(overlay).save(out_path)

print(f"완료! overlay 이미지는 여기 저장됨: {out_dir}")


In [ ]:
from PIL import Image
import numpy as np

# 이미지 경로
img_path = "results/BP_result_image_30/target/image_0276.png"

# 이미지 불러오기
img = Image.open(img_path).convert("RGB")
img_np = np.array(img)

# 빨간색 픽셀 조건 (정확한 빨강)
red_mask = (img_np[:, :, 0] == 255) & (img_np[:, :, 1] == 0) & (img_np[:, :, 2] == 0)

# 빨간색 → 흰색 변경
img_np[red_mask] = [255, 255, 255]

# 변경된 이미지 저장
output_path = "results/BP_result_image_30/target/image_0276_white.png"
Image.fromarray(img_np).save(output_path)

print("변환 완료:", output_path)


In [ ]:
import cv2
import numpy as np

# ===== 파일 경로 =====
input_path = "results/result_image_30/input/image_0116.png"
gt_path    = "results/result_image_30/target/image_0116.png"
save_path  = "results/result_image_30/overlay_116.png"

alpha = 0.45  # 투명도

# ===== 이미지 읽기 =====
img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
gt  = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE)

# 크기 맞추기
if gt.shape != img.shape:
    gt = cv2.resize(gt, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)

# grayscale → BGR
overlay = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

# mask 영역
mask = gt > 0

# 초록색 만들기
green = np.zeros_like(overlay)
green[:, :, 1] = 255

# overlay 적용
overlay[mask] = cv2.addWeighted(
    overlay[mask],
    1 - alpha,
    green[mask],
    alpha,
    0
)

# 저장
cv2.imwrite(save_path, overlay)

print("저장 완료:", save_path)